# TP OmniAI - máy chủ AI chạy trên Colab
Notebook này bật **OmniRoute** và **backend bảo mật** trong phiên Colab của bạn, rồi mở đường hầm **ngrok** để website trên GitHub Pages kết nối vào.

**Mỗi lần muốn dùng:** chạy lần lượt *Bước 1* rồi *Bước 2* (bấm nút ▶ ở mỗi ô). Khi xong việc, chạy ô *Tắt dịch vụ*.

**Cần biết:**
- Colab không chạy 24/7: phiên có thể bị ngắt khi nhàn rỗi hoặc hết thời gian. Khi đó website vẫn mở được nhưng AI sẽ báo mất kết nối; bạn chỉ cần chạy lại Bước 1 và 2.
- Colab hạn chế chạy "dịch vụ web không liên quan đến tính toán tương tác" và có thể ngắt phiên miễn phí không báo trước. Hãy dùng ở quy mô nhỏ (bạn bè).

In [ ]:
#@title Bước 1 - Cài đặt (chạy 1 lần mỗi phiên Colab, mất vài phút) {display-mode: "form"}
OMNIROUTE_VERSION = "3.8.52"  #@param {type:"string"}
import pathlib, sys
BASE = pathlib.Path("/content/tp-omniai"); (BASE/"backend").mkdir(parents=True, exist_ok=True)
SERVER_MJS = r'''// TP OmniAI — backend trung gian (không cần cài thêm thư viện, Node.js >= 20).
//
// Vai trò:
//   Trình duyệt (GitHub Pages) -> backend này -> OmniRoute (chỉ nội bộ, 127.0.0.1)
// Backend xác thực mật khẩu dùng chung, cấp phiên có thời hạn, giới hạn tần suất,
// kiểm tra dữ liệu đầu vào rồi chuyển tiếp tới OmniRoute. Khóa API (nếu có) chỉ nằm ở đây.
//
// Cấu hình hoàn toàn bằng biến môi trường (xem .env.example).

import http from 'node:http';
import crypto from 'node:crypto';
import { Readable } from 'node:stream';

const env = process.env;
const num = (v, d) => (v !== undefined && v !== '' && Number.isFinite(Number(v)) ? Number(v) : d);

const cfg = {
  host: env.HOST || '127.0.0.1',
  port: num(env.PORT, 8080),
  omniUrl: (env.OMNIROUTE_URL || 'http://127.0.0.1:20128').replace(/\/+$/, ''),
  omniKey: env.OMNIROUTE_API_KEY || '',
  sitePassword: env.SITE_PASSWORD || '',
  sessionSecret: env.SESSION_SECRET || crypto.randomBytes(32).toString('hex'),
  sessionTtlMs: num(env.SESSION_TTL_MINUTES, 720) * 60_000,
  allowedOrigins: (env.ALLOWED_ORIGIN || '')
    .split(',')
    .map((s) => s.trim().replace(/\/+$/, ''))
    .filter(Boolean),
  maxBodyBytes: num(env.MAX_BODY_BYTES, 1_500_000),
  loginMaxFailures: num(env.LOGIN_MAX_FAILURES, 5),
  loginWindowMs: num(env.LOGIN_WINDOW_MINUTES, 15) * 60_000,
  chatPerMinute: num(env.CHAT_PER_MINUTE, 20),
  maxConcurrentChats: num(env.MAX_CONCURRENT_CHATS, 6),
  upstreamTimeoutMs: num(env.UPSTREAM_TIMEOUT_SECONDS, 120) * 1000,
  streamIdleMs: num(env.STREAM_IDLE_SECONDS, 90) * 1000,
};

// ---------- Kiểm tra cấu hình khi khởi động (thất bại thì dừng, không chạy ở chế độ lỏng lẻo) ----------
function fatal(msg) {
  console.error('[TP OmniAI] Lỗi cấu hình: ' + msg);
  process.exit(1);
}
if (cfg.sitePassword.length < 8) fatal('SITE_PASSWORD phải có ít nhất 8 ký tự.');
if (cfg.allowedOrigins.length === 0) fatal('ALLOWED_ORIGIN chưa được đặt (ví dụ https://tenban.github.io).');
for (const o of cfg.allowedOrigins) {
  let u;
  try {
    u = new URL(o);
  } catch {
    fatal('ALLOWED_ORIGIN không hợp lệ: ' + o);
  }
  const local = u.hostname === 'localhost' || u.hostname === '127.0.0.1';
  if (u.origin !== o || !(u.protocol === 'https:' || (local && u.protocol === 'http:'))) {
    fatal('ALLOWED_ORIGIN phải có dạng https://tenmien (không có đường dẫn phía sau): ' + o);
  }
}
try {
  const u = new URL(cfg.omniUrl);
  if (!['127.0.0.1', 'localhost', '::1'].includes(u.hostname) && env.ALLOW_REMOTE_ROUTER !== '1') {
    fatal('OMNIROUTE_URL phải trỏ về máy này (127.0.0.1). Đây là biện pháp chống chuyển tiếp tới địa chỉ tùy ý.');
  }
} catch {
  fatal('OMNIROUTE_URL không hợp lệ.');
}
const allowedOrigins = new Set(cfg.allowedOrigins);

// ---------- Tiện ích ----------
const EXPOSED_HEADERS = ['X-OmniRoute-Decision', 'X-OmniRoute-Model', 'X-OmniRoute-Provider', 'X-OmniRoute-Fallback-Attempts'];
const SECRET_PATTERNS = [/sk-[A-Za-z0-9_\-]{8,}/g, /Bearer\s+[A-Za-z0-9._\-]{8,}/gi, /[A-Za-z0-9_\-]{32,}/g];

function redact(text) {
  let out = String(text ?? '');
  for (const re of SECRET_PATTERNS) out = out.replace(re, '[đã ẩn]');
  return out.replace(/[\u0000-\u001f\u007f]/g, ' ').slice(0, 300);
}
function cleanHeaderValue(v) {
  return String(v).replace(/[^\x20-\x7e]/g, '').slice(0, 200);
}
function log(req, status, started) {
  // Chỉ ghi phương thức, đường dẫn và mã trạng thái. Không ghi nội dung, khóa hay mật khẩu.
  const path = (req.url || '').split('?')[0];
  console.log(`${new Date().toISOString()} ${req.method} ${path} ${status} ${Date.now() - started}ms`);
}

class SlidingWindow {
  constructor(max, windowMs) {
    this.max = max;
    this.windowMs = windowMs;
    this.hits = new Map();
  }
  _prune(key, now) {
    const arr = (this.hits.get(key) || []).filter((t) => now - t < this.windowMs);
    if (arr.length) this.hits.set(key, arr);
    else this.hits.delete(key);
    return arr;
  }
  count(key) {
    return this._prune(key, Date.now()).length;
  }
  add(key) {
    const now = Date.now();
    const arr = this._prune(key, now);
    arr.push(now);
    this.hits.set(key, arr);
  }
  retryAfterSec(key) {
    const arr = this._prune(key, Date.now());
    if (!arr.length) return 0;
    return Math.max(1, Math.ceil((arr[0] + this.windowMs - Date.now()) / 1000));
  }
  sweep() {
    const now = Date.now();
    for (const k of [...this.hits.keys()]) this._prune(k, now);
  }
}
const loginFailuresByIp = new SlidingWindow(cfg.loginMaxFailures, cfg.loginWindowMs);
const loginFailuresGlobal = new SlidingWindow(cfg.loginMaxFailures * 6, cfg.loginWindowMs);
const chatRate = new SlidingWindow(cfg.chatPerMinute, 60_000);
setInterval(() => {
  loginFailuresByIp.sweep();
  loginFailuresGlobal.sweep();
  chatRate.sweep();
}, 60_000).unref();

function clientIp(req) {
  // Backend chỉ nhận kết nối từ ngrok (cùng máy). Dòng cuối của X-Forwarded-For do ngrok thêm vào.
  const xff = req.headers['x-forwarded-for'];
  if (typeof xff === 'string' && xff.trim()) {
    const parts = xff.split(',').map((s) => s.trim()).filter(Boolean);
    return parts[parts.length - 1] || 'unknown';
  }
  return req.socket.remoteAddress || 'unknown';
}

// ---------- Phiên đăng nhập (token ký HMAC, có hạn dùng) ----------
const hmac = (data) => crypto.createHmac('sha256', cfg.sessionSecret).update(data).digest();

function signToken() {
  const now = Date.now();
  const payload = Buffer.from(JSON.stringify({ iat: now, exp: now + cfg.sessionTtlMs, n: crypto.randomBytes(8).toString('hex') })).toString('base64url');
  return { token: payload + '.' + hmac(payload).toString('base64url'), exp: now + cfg.sessionTtlMs };
}
function verifyToken(token) {
  if (typeof token !== 'string' || token.length > 512) return null;
  const parts = token.split('.');
  if (parts.length !== 2 || !parts[0] || !parts[1]) return null;
  let given;
  try {
    given = Buffer.from(parts[1], 'base64url');
  } catch {
    return null;
  }
  const expected = hmac(parts[0]);
  if (given.length !== expected.length || !crypto.timingSafeEqual(given, expected)) return null;
  try {
    const data = JSON.parse(Buffer.from(parts[0], 'base64url').toString('utf8'));
    if (typeof data.exp !== 'number' || data.exp <= Date.now()) return null;
    return data;
  } catch {
    return null;
  }
}
function passwordMatches(candidate) {
  const a = crypto.createHash('sha256').update(String(candidate)).digest();
  const b = crypto.createHash('sha256').update(cfg.sitePassword).digest();
  return crypto.timingSafeEqual(a, b);
}
function authenticate(req) {
  const h = req.headers.authorization;
  if (typeof h !== 'string' || !h.startsWith('Bearer ')) return null;
  return verifyToken(h.slice(7).trim());
}

// ---------- Phản hồi ----------
function baseHeaders(req) {
  const h = {
    'Cache-Control': 'no-store',
    'X-Content-Type-Options': 'nosniff',
    'Referrer-Policy': 'no-referrer',
    Vary: 'Origin',
  };
  const origin = req.headers.origin;
  if (origin && allowedOrigins.has(origin)) {
    h['Access-Control-Allow-Origin'] = origin;
    h['Access-Control-Allow-Headers'] = 'Content-Type, Authorization, ngrok-skip-browser-warning';
    h['Access-Control-Allow-Methods'] = 'GET, POST, OPTIONS';
    h['Access-Control-Expose-Headers'] = [...EXPOSED_HEADERS, 'Retry-After'].join(', ');
    h['Access-Control-Max-Age'] = '600';
  }
  return h;
}
function sendJson(req, res, status, body, extra = {}) {
  if (res.headersSent) return res.end();
  const payload = JSON.stringify(body);
  res.writeHead(status, { ...baseHeaders(req), 'Content-Type': 'application/json; charset=utf-8', 'Content-Length': Buffer.byteLength(payload), ...extra });
  res.end(payload);
}
function sendError(req, res, status, code, message, extra = {}, detail) {
  const body = { error: { code, message } };
  if (detail) body.error.detail = redact(detail);
  sendJson(req, res, status, body, extra);
}

async function readJson(req) {
  const type = String(req.headers['content-type'] || '');
  if (!type.toLowerCase().startsWith('application/json')) throw Object.assign(new Error('Content-Type must be application/json'), { code: 'bad_request' });
  const declared = Number(req.headers['content-length']);
  if (Number.isFinite(declared) && declared > cfg.maxBodyBytes) throw Object.assign(new Error('Request too large'), { code: 'payload_too_large' });
  const chunks = [];
  let size = 0;
  for await (const chunk of req) {
    size += chunk.length;
    if (size > cfg.maxBodyBytes) throw Object.assign(new Error('Request too large'), { code: 'payload_too_large' });
    chunks.push(chunk);
  }
  try {
    const parsed = JSON.parse(Buffer.concat(chunks).toString('utf8'));
    if (parsed === null || typeof parsed !== 'object' || Array.isArray(parsed)) throw new Error('not an object');
    return parsed;
  } catch {
    throw Object.assign(new Error('Invalid JSON'), { code: 'bad_request' });
  }
}

// ---------- Gọi OmniRoute ----------
function routerHeaders(extra = {}) {
  const h = { Accept: 'application/json', ...extra };
  if (cfg.omniKey) h.Authorization = 'Bearer ' + cfg.omniKey;
  return h;
}
function classifyFetchError(err) {
  const name = err?.name;
  const code = err?.cause?.code || err?.code;
  if (name === 'AbortError' || name === 'TimeoutError') return { status: 504, code: 'router_timeout', message: 'OmniRoute did not respond in time.' };
  if (['ECONNREFUSED', 'ECONNRESET', 'EHOSTUNREACH', 'ENOTFOUND', 'UND_ERR_SOCKET', 'UND_ERR_CONNECT_TIMEOUT'].includes(code) || name === 'TypeError')
    return { status: 503, code: 'router_unreachable', message: 'Cannot reach OmniRoute.' };
  return { status: 502, code: 'provider_error', message: 'Unexpected error while contacting OmniRoute.' };
}
async function upstreamError(upstream) {
  let text = '';
  try {
    text = (await upstream.text()).slice(0, 4000);
  } catch {
    /* bỏ qua */
  }
  let message = text;
  try {
    const j = JSON.parse(text);
    message = j?.error?.message || j?.message || (typeof j?.error === 'string' ? j.error : text);
  } catch {
    /* không phải JSON */
  }
  const s = upstream.status;
  const retry = upstream.headers.get('retry-after');
  const extra = retry && /^\d+$/.test(retry) ? { 'Retry-After': retry } : {};
  if (s === 401 || s === 403) return { status: 502, code: 'router_auth', msg: 'OmniRoute rejected the access key.', extra, message };
  if (s === 404) return { status: 404, code: /model/i.test(message) ? 'model_not_found' : 'upstream_not_found', msg: 'Model or route not found.', extra, message };
  if (s === 429) return { status: 429, code: /quota|credit|exhaust|billing|insufficient/i.test(message) ? 'quota_exhausted' : 'rate_limited', msg: 'Provider limit reached.', extra, message };
  if (s === 400 || s === 413 || s === 422) return { status: 400, code: 'bad_request', msg: 'The router rejected the request.', extra, message };
  return { status: 502, code: 'provider_error', msg: 'The provider returned an error.', extra, message };
}
function copyRouterHeaders(upstream) {
  const out = {};
  for (const name of EXPOSED_HEADERS) {
    const v = upstream.headers.get(name);
    if (v) out[name] = cleanHeaderValue(v);
  }
  return out;
}

// ---------- Kiểm tra dữ liệu chat ----------
const MODEL_RE = /^(?!.*(?:\.\.|:\/\/))[A-Za-z0-9][A-Za-z0-9._:/@+\-]{0,199}$/;
function validateChat(body) {
  if (typeof body.model !== 'string' || !MODEL_RE.test(body.model)) return 'model is missing or invalid';
  if (!Array.isArray(body.messages) || body.messages.length < 1 || body.messages.length > 200) return 'messages must contain 1-200 items';
  for (const m of body.messages) {
    if (!m || typeof m !== 'object') return 'invalid message';
    if (!['system', 'user', 'assistant'].includes(m.role)) return 'invalid role';
    if (typeof m.content !== 'string' || m.content.length === 0 || m.content.length > 600_000) return 'invalid content';
  }
  if (body.temperature !== undefined && !(typeof body.temperature === 'number' && body.temperature >= 0 && body.temperature <= 2)) return 'invalid temperature';
  if (body.max_tokens !== undefined && !(Number.isInteger(body.max_tokens) && body.max_tokens >= 1 && body.max_tokens <= 65536)) return 'invalid max_tokens';
  if (body.stream !== undefined && typeof body.stream !== 'boolean') return 'invalid stream';
  return null;
}

// ---------- Các endpoint ----------
let activeChats = 0;

async function handleLogin(req, res) {
  const ip = clientIp(req);
  if (loginFailuresByIp.count(ip) >= cfg.loginMaxFailures || loginFailuresGlobal.count('*') >= cfg.loginMaxFailures * 6) {
    const wait = Math.max(loginFailuresByIp.retryAfterSec(ip), loginFailuresGlobal.retryAfterSec('*'));
    return sendError(req, res, 429, 'too_many_attempts', 'Too many failed attempts. Try again later.', { 'Retry-After': String(wait) });
  }
  const body = await readJson(req);
  if (typeof body.password !== 'string' || body.password.length === 0 || body.password.length > 200 || !passwordMatches(body.password)) {
    loginFailuresByIp.add(ip);
    loginFailuresGlobal.add('*');
    await new Promise((r) => setTimeout(r, 400));
    return sendError(req, res, 401, 'bad_credentials', 'Wrong password.');
  }
  const { token, exp } = signToken();
  sendJson(req, res, 200, { token, expiresAt: exp });
}

async function handleModels(req, res) {
  let upstream;
  try {
    upstream = await fetch(cfg.omniUrl + '/v1/models?prefix=alias', { headers: routerHeaders(), signal: AbortSignal.timeout(20_000) });
  } catch (err) {
    const e = classifyFetchError(err);
    return sendError(req, res, e.status, e.code, e.message);
  }
  if (!upstream.ok) {
    const e = await upstreamError(upstream);
    return sendError(req, res, e.status, e.code, e.msg, e.extra, e.message);
  }
  let json;
  try {
    json = await upstream.json();
  } catch {
    return sendError(req, res, 502, 'bad_upstream_response', 'OmniRoute returned an unreadable model list.');
  }
  if (!json || !Array.isArray(json.data)) return sendError(req, res, 502, 'bad_upstream_response', 'OmniRoute returned an unexpected model list.');
  const models = json.data
    .filter((m) => m && typeof m.id === 'string' && MODEL_RE.test(m.id))
    .slice(0, 3000)
    .map((m) => ({ id: m.id, owned_by: typeof m.owned_by === 'string' ? m.owned_by.slice(0, 80) : undefined, type: typeof m.type === 'string' ? m.type.slice(0, 40) : undefined }));
  sendJson(req, res, 200, { models, total: models.length });
}

async function handleStatus(req, res, url) {
  const out = { backend: { ok: true }, router: { reachable: false } };
  const t0 = Date.now();
  try {
    const r = await fetch(cfg.omniUrl + '/v1/models?prefix=alias', { headers: routerHeaders(), signal: AbortSignal.timeout(10_000) });
    out.router.httpStatus = r.status;
    out.router.latencyMs = Date.now() - t0;
    if (r.ok) {
      out.router.reachable = true;
      try {
        const j = await r.json();
        out.router.modelCount = Array.isArray(j.data) ? j.data.length : null;
      } catch {
        out.router.modelCount = null;
      }
    } else {
      const e = await upstreamError(r);
      out.router.error = { code: e.code, detail: redact(e.message) };
    }
  } catch (err) {
    const e = classifyFetchError(err);
    out.router.error = { code: e.code };
  }
  const probeModel = url.searchParams.get('probe');
  if (probeModel && out.router.reachable) {
    if (!MODEL_RE.test(probeModel)) return sendError(req, res, 400, 'bad_request', 'Invalid model.');
    const p0 = Date.now();
    try {
      const r = await fetch(cfg.omniUrl + '/v1/chat/completions', {
        method: 'POST',
        headers: routerHeaders({ 'Content-Type': 'application/json' }),
        body: JSON.stringify({ model: probeModel, messages: [{ role: 'user', content: 'ping' }], max_tokens: 8, stream: false }),
        signal: AbortSignal.timeout(60_000),
      });
      const hdr = copyRouterHeaders(r);
      if (r.ok) {
        let reported = null;
        try {
          reported = (await r.json())?.model ?? null;
        } catch {
          /* bỏ qua */
        }
        out.probe = { model: probeModel, ok: true, latencyMs: Date.now() - p0, reportedModel: typeof reported === 'string' ? reported.slice(0, 200) : null, headers: hdr };
      } else {
        const e = await upstreamError(r);
        out.probe = { model: probeModel, ok: false, latencyMs: Date.now() - p0, error: { code: e.code, detail: redact(e.message) } };
      }
    } catch (err) {
      out.probe = { model: probeModel, ok: false, error: { code: classifyFetchError(err).code } };
    }
  }
  sendJson(req, res, 200, out);
}

async function handleChat(req, res, auth) {
  if (chatRate.count(auth.n) >= cfg.chatPerMinute) {
    return sendError(req, res, 429, 'rate_limited', 'Too many requests. Slow down.', { 'Retry-After': String(chatRate.retryAfterSec(auth.n)) });
  }
  if (activeChats >= cfg.maxConcurrentChats) return sendError(req, res, 503, 'server_busy', 'The server is busy. Try again shortly.', { 'Retry-After': '5' });
  const body = await readJson(req);
  const problem = validateChat(body);
  if (problem) return sendError(req, res, 400, 'bad_request', problem);
  chatRate.add(auth.n);

  const stream = body.stream !== false;
  const payload = { model: body.model, messages: body.messages.map((m) => ({ role: m.role, content: m.content })), stream };
  if (body.temperature !== undefined) payload.temperature = body.temperature;
  if (body.max_tokens !== undefined) payload.max_tokens = body.max_tokens;

  const controller = new AbortController();
  const onClientGone = () => {
    if (!res.writableEnded) controller.abort();
  };
  res.on('close', onClientGone);
  const headerTimer = setTimeout(() => controller.abort(), cfg.upstreamTimeoutMs);
  activeChats++;
  let released = false;
  const release = () => {
    if (!released) {
      released = true;
      activeChats--;
    }
  };
  res.on('close', release);

  let upstream;
  try {
    upstream = await fetch(cfg.omniUrl + '/v1/chat/completions', {
      method: 'POST',
      headers: routerHeaders({ 'Content-Type': 'application/json', Accept: stream ? 'text/event-stream' : 'application/json' }),
      body: JSON.stringify(payload),
      signal: controller.signal,
    });
  } catch (err) {
    clearTimeout(headerTimer);
    if (res.destroyed) return;
    const e = classifyFetchError(err);
    return sendError(req, res, e.status, e.code, e.message);
  }
  clearTimeout(headerTimer);

  if (!upstream.ok) {
    const e = await upstreamError(upstream);
    return sendError(req, res, e.status, e.code, e.msg, e.extra, e.message);
  }

  const routerHdrs = copyRouterHeaders(upstream);
  const ctype = upstream.headers.get('content-type') || '';
  if (!stream || !ctype.includes('text/event-stream')) {
    // Phản hồi thường (không streaming) hoặc router trả JSON dù yêu cầu stream.
    const text = await upstream.text();
    res.writeHead(200, { ...baseHeaders(req), ...routerHdrs, 'Content-Type': ctype.includes('json') ? 'application/json; charset=utf-8' : 'text/plain; charset=utf-8' });
    return res.end(text);
  }

  res.writeHead(200, { ...baseHeaders(req), ...routerHdrs, 'Content-Type': 'text/event-stream; charset=utf-8', 'Cache-Control': 'no-store, no-transform', 'X-Accel-Buffering': 'no' });
  res.socket?.setNoDelay(true);
  res.flushHeaders();

  const nodeStream = Readable.fromWeb(upstream.body);
  let idle;
  const armIdle = () => {
    clearTimeout(idle);
    idle = setTimeout(() => controller.abort(), cfg.streamIdleMs);
  };
  armIdle();
  nodeStream.on('data', armIdle);
  nodeStream.on('error', () => {
    clearTimeout(idle);
    // Luồng bị ngắt giữa chừng: kết thúc kết nối đột ngột để trình duyệt nhận biết là chưa hoàn tất.
    res.destroy();
  });
  nodeStream.on('end', () => clearTimeout(idle));
  res.on('close', () => {
    clearTimeout(idle);
    nodeStream.destroy();
  });
  nodeStream.pipe(res);
}

const server = http.createServer(async (req, res) => {
  const started = Date.now();
  res.on('finish', () => log(req, res.statusCode, started));
  res.on('close', () => {
    if (!res.writableFinished) log(req, res.statusCode + '(closed)', started);
  });
  try {
    const url = new URL(req.url, 'http://localhost');
    const origin = req.headers.origin;
    if (origin && !allowedOrigins.has(origin)) return sendError(req, res, 403, 'origin_not_allowed', 'Origin not allowed.');

    if (req.method === 'OPTIONS') {
      res.writeHead(204, baseHeaders(req));
      return res.end();
    }
    if (url.pathname === '/' && req.method === 'GET') return sendJson(req, res, 200, { service: 'TP OmniAI backend', ok: true });
    if (url.pathname === '/api/health' && req.method === 'GET') return sendJson(req, res, 200, { ok: true, service: 'tp-omniai', time: new Date().toISOString() });
    if (url.pathname === '/api/login' && req.method === 'POST') return await handleLogin(req, res);

    if (url.pathname.startsWith('/api/')) {
      const auth = authenticate(req);
      if (!auth) return sendError(req, res, 401, 'unauthorized', 'Login required or session expired.');
      if (url.pathname === '/api/models' && req.method === 'GET') return await handleModels(req, res);
      if (url.pathname === '/api/status' && req.method === 'GET') return await handleStatus(req, res, url);
      if (url.pathname === '/api/chat' && req.method === 'POST') return await handleChat(req, res, auth);
    }
    return sendError(req, res, 404, 'not_found', 'Not found.');
  } catch (err) {
    if (err?.code === 'payload_too_large') return sendError(req, res, 413, 'payload_too_large', 'Request too large.');
    if (err?.code === 'bad_request') return sendError(req, res, 400, 'bad_request', err.message);
    console.error('[TP OmniAI] Lỗi nội bộ:', err?.name || 'Error'); // không in stack/nội dung
    sendError(req, res, 500, 'internal_error', 'Internal error.');
  }
});
server.headersTimeout = 20_000;
server.requestTimeout = 0; // luồng streaming có thể kéo dài; đã có giới hạn riêng theo từng bước
server.keepAliveTimeout = 5_000;

server.listen(cfg.port, cfg.host, () => {
  console.log(`[TP OmniAI] Backend đang chạy tại http://${cfg.host}:${cfg.port} → OmniRoute ${cfg.omniUrl}`);
  console.log(`[TP OmniAI] Cho phép nguồn truy cập: ${cfg.allowedOrigins.join(', ')}`);
});
for (const sig of ['SIGINT', 'SIGTERM']) process.on(sig, () => server.close(() => process.exit(0)));
'''

TP_HELPER = r'''"""TP OmniAI - tiện ích chạy trong Google Colab: cài đặt, khởi động OmniRoute + backend + ngrok.

Được notebook gọi; người dùng không cần sửa tệp này.
Không in mật khẩu/token ra màn hình; mọi nội dung log đều được ẩn bí mật trước khi hiển thị.
"""
import getpass
import json
import os
import pathlib
import re
import shutil
import signal
import socket
import sqlite3
import subprocess
import sys
import tarfile
import threading
import time
import urllib.error
import urllib.parse
import urllib.request

BASE = pathlib.Path(os.environ.get("TP_BASE", "/content/tp-omniai"))
DATA_DIR = BASE / "omniroute-data"
BACKEND_DIR = BASE / "backend"
LOG_DIR = BASE / "logs"
DRIVE_DIR = pathlib.Path(os.environ.get("TP_DRIVE_DIR", "/content/drive/MyDrive/TP_OmniAI"))
OMNI_PORT = int(os.environ.get("TP_OMNI_PORT", "20128"))
BACK_PORT = int(os.environ.get("TP_BACK_PORT", "8080"))
NGROK_API = os.environ.get("TP_NGROK_API", "http://127.0.0.1:4040")
NGROK_URL = "https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz"

_procs = {}
_secrets = set()
_state = {"persist": False, "origin": "", "url": ""}


class SetupError(RuntimeError):
    """Lỗi đã được giải thích bằng tiếng Việt; notebook chỉ cần hiển thị thông điệp."""


def say(msg=""):
    print(msg, flush=True)


def _register_secret(value):
    if value and len(value) >= 6:
        _secrets.add(value)


def redact(text):
    for s in sorted(_secrets, key=len, reverse=True):
        text = text.replace(s, "[đã ẩn]")
    return text


def _tail(path, n=30):
    try:
        lines = pathlib.Path(path).read_text(errors="replace").splitlines()[-n:]
    except OSError:
        return "(chưa có log)"
    return redact("\n".join(lines))


def _sh(cmd, log_name, what):
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    log = LOG_DIR / log_name
    with open(log, "ab") as f:
        r = subprocess.run(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT)
    if r.returncode != 0:
        raise SetupError(f"{what} thất bại (mã {r.returncode}). 30 dòng log cuối:\n{_tail(log)}")


# ---------------------------------------------------------------- Cài đặt
def node_version():
    try:
        out = subprocess.run(["node", "-v"], capture_output=True, text=True).stdout.strip()
    except FileNotFoundError:
        return None
    m = re.match(r"v(\d+)\.(\d+)\.(\d+)", out)
    return tuple(int(x) for x in m.groups()) if m else None


def node_ok(v):
    # Theo package.json của OmniRoute 3.8.52: >=22.22.2 <23 hoặc >=24.0.0 <27
    return bool(v) and ((v >= (22, 22, 2) and v < (23, 0, 0)) or (v >= (24, 0, 0) and v < (27, 0, 0)))


def install_runtime(omniroute_version="3.8.52"):
    BASE.mkdir(parents=True, exist_ok=True)
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    if not (BACKEND_DIR / "server.mjs").exists():
        raise SetupError("Thiếu tệp backend/server.mjs. Hãy chạy lại ô cài đặt từ đầu.")

    v = node_version()
    if not node_ok(v):
        for major in (22, 24):
            say(f"Đang cài Node.js {major} (OmniRoute yêu cầu Node 22.22.2+ hoặc 24+)...")
            _sh(f"curl -fsSL https://deb.nodesource.com/setup_{major}.x | bash - && apt-get install -y nodejs", "node.log", f"Cài Node.js {major}")
            v = node_version()
            if node_ok(v):
                break
        if not node_ok(v):
            raise SetupError(f"Không cài được phiên bản Node.js phù hợp (hiện có: {v}).")
    say("Node.js: v" + ".".join(map(str, v)) + " (đạt yêu cầu)")

    say(f"Đang cài OmniRoute {omniroute_version} (có thể mất vài phút)...")
    try:
        _sh(f"npm install -g omniroute@{omniroute_version}", "npm-omniroute.log", f"Cài OmniRoute {omniroute_version}")
    except SetupError as first:
        say(f"Không cài được đúng phiên bản {omniroute_version}. Thử bản mới nhất (có thể khác bản đã được khảo sát)...")
        try:
            _sh("npm install -g omniroute@latest", "npm-omniroute.log", "Cài OmniRoute (bản mới nhất)")
        except SetupError:
            raise first
    prefix = subprocess.run(["npm", "prefix", "-g"], capture_output=True, text=True).stdout.strip()
    os.environ["PATH"] = f"{prefix}/bin:" + os.environ["PATH"]
    if not shutil.which("omniroute"):
        raise SetupError("Đã cài xong nhưng không tìm thấy lệnh 'omniroute'.")
    ver = subprocess.run(["npm", "ls", "-g", "omniroute", "--depth=0"], capture_output=True, text=True).stdout.strip().splitlines()
    say("OmniRoute đã cài: " + (ver[-1].strip() if ver else "?"))

    if not shutil.which("ngrok"):
        say("Đang tải ngrok...")
        try:
            tgz = BASE / "ngrok.tgz"
            urllib.request.urlretrieve(NGROK_URL, tgz)
            with tarfile.open(tgz) as t:
                t.extract("ngrok", "/usr/local/bin")
            os.chmod("/usr/local/bin/ngrok", 0o755)
        except Exception as e:  # noqa: BLE001
            raise SetupError(f"Không tải được ngrok ({type(e).__name__}). Kiểm tra kết nối mạng của Colab rồi chạy lại.")
    say("ngrok: " + subprocess.run(["ngrok", "version"], capture_output=True, text=True).stdout.strip())
    say("\nCài đặt xong. Chạy tiếp ô 'Khởi động'.")


# ---------------------------------------------------------------- Bí mật & lưu trữ
def _ask(name, prompt, min_len=0):
    for _ in range(3):
        value = None
        try:
            from google.colab import userdata  # Colab Secrets (biểu tượng chìa khóa bên trái)

            value = userdata.get(name)
        except Exception:  # noqa: BLE001 - chưa có secret hoặc không chạy trên Colab
            value = None
        value = value or os.environ.get(name) or getpass.getpass(prompt + ": ")
        value = value.strip()
        if len(value) >= max(min_len, 1):
            _register_secret(value)
            return value
        say(f"  Cần ít nhất {min_len} ký tự, nhập lại.")
    raise SetupError(f"Chưa nhập {name} hợp lệ.")


def mount_drive():
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)


def _persistent_secrets(persist):
    folder = DRIVE_DIR if persist else BASE
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / "secrets.json"
    try:
        data = json.loads(path.read_text())
    except (OSError, ValueError):
        data = {}
    import secrets as _s

    changed = False
    for k in ("JWT_SECRET", "API_KEY_SECRET"):
        if not data.get(k):
            data[k] = _s.token_hex(32)
            changed = True
    if changed:
        path.write_text(json.dumps(data))
        os.chmod(path, 0o600)
    for v in data.values():
        _register_secret(v)
    return data


def _link_home_dir():
    """Nếu OmniRoute dùng ~/.omniroute thay vì DATA_DIR, liên kết sang DATA_DIR để dữ liệu vẫn nằm đúng chỗ."""
    home = pathlib.Path.home() / ".omniroute"
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    if home.is_symlink():
        home.unlink()
    if not home.exists():
        home.symlink_to(DATA_DIR, target_is_directory=True)
    elif home.is_dir() and not any(home.iterdir()):
        home.rmdir()
        home.symlink_to(DATA_DIR, target_is_directory=True)


def restore_config():
    src = DRIVE_DIR / "omniroute-data"
    if not src.exists() or not any(src.rglob("*")):
        return False
    if any(DATA_DIR.rglob("*.db")):
        return False
    shutil.copytree(src, DATA_DIR, dirs_exist_ok=True)
    say("Đã khôi phục cấu hình OmniRoute từ Google Drive.")
    return True


def backup_config():
    """Sao lưu dữ liệu OmniRoute sang Google Drive. Cơ sở dữ liệu SQLite được sao lưu bằng API sao lưu an toàn (không chép tệp đang mở)."""
    if not DRIVE_DIR.parent.exists():
        say("Chưa gắn Google Drive (đặt PERSIST_TO_DRIVE = True ở ô Khởi động).")
        return 0
    target = DRIVE_DIR / "omniroute-data"
    count = 0
    for p in DATA_DIR.rglob("*"):
        if not p.is_file() or p.name.endswith(("-wal", "-shm", "-journal")):
            continue
        dest = target / p.relative_to(DATA_DIR)
        dest.parent.mkdir(parents=True, exist_ok=True)
        if p.suffix in (".db", ".sqlite", ".sqlite3"):
            src = sqlite3.connect(str(p))
            dst = sqlite3.connect(str(dest))
            try:
                src.backup(dst)
            finally:
                dst.close()
                src.close()
        elif p.stat().st_size < 20_000_000:
            shutil.copy2(p, dest)
        else:
            continue
        count += 1
    say(f"Đã sao lưu {count} tệp cấu hình vào Google Drive ({target}).")
    return count


def _auto_backup_loop():
    while _state.get("persist"):
        time.sleep(600)
        try:
            backup_config()
        except Exception:  # noqa: BLE001
            pass


# ---------------------------------------------------------------- Tiến trình
def _port_open(port):
    with socket.socket() as s:
        s.settimeout(0.5)
        return s.connect_ex(("127.0.0.1", port)) == 0


def _pids_listening(port):
    """Các PID đang lắng nghe cổng TCP `port` (đọc /proc, không cần công cụ ngoài)."""
    inodes = set()
    for table in ("/proc/net/tcp", "/proc/net/tcp6"):
        try:
            lines = pathlib.Path(table).read_text().splitlines()[1:]
        except OSError:
            continue
        for line in lines:
            cols = line.split()
            if len(cols) > 9 and cols[3] == "0A" and int(cols[1].split(":")[1], 16) == port:
                inodes.add(cols[9])
    pids = set()
    if not inodes:
        return pids
    for proc in pathlib.Path("/proc").iterdir():
        if not proc.name.isdigit():
            continue
        try:
            for fd in (proc / "fd").iterdir():
                if os.readlink(fd).startswith("socket:[") and os.readlink(fd)[8:-1] in inodes:
                    pids.add(int(proc.name))
                    break
        except OSError:
            continue
    pids.discard(os.getpid())
    return pids


def _wait_port(port, proc, name, log, timeout=150):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if proc.poll() is not None:
            raise SetupError(f"{name} dừng đột ngột (mã {proc.returncode}). Log cuối:\n{_tail(log)}")
        if _port_open(port):
            return
        time.sleep(1)
    raise SetupError(f"{name} không mở cổng {port} sau {timeout}s. Log cuối:\n{_tail(log)}")


def _spawn(name, cmd, env, log_name):
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    log = LOG_DIR / log_name
    f = open(log, "wb")
    p = subprocess.Popen(cmd, env=env, cwd=str(BASE), stdout=f, stderr=subprocess.STDOUT, start_new_session=True)
    _procs[name] = p
    return p, log


def _kill(name):
    p = _procs.pop(name, None)
    if p and p.poll() is None:
        try:
            os.killpg(os.getpgid(p.pid), signal.SIGTERM)
            p.wait(timeout=8)
        except Exception:  # noqa: BLE001
            try:
                os.killpg(os.getpgid(p.pid), signal.SIGKILL)
            except Exception:  # noqa: BLE001
                pass


def start_omniroute(admin_password, persist):
    if "omniroute" in _procs and _procs["omniroute"].poll() is None and _port_open(OMNI_PORT):
        say("OmniRoute đã chạy sẵn.")
        return
    _kill("omniroute")
    if _port_open(OMNI_PORT):
        raise SetupError(f"Cổng {OMNI_PORT} đang bị chiếm bởi một tiến trình khác. Chạy ô 'Tắt dịch vụ' rồi thử lại.")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    _link_home_dir()
    if persist:
        restore_config()
    sec = _persistent_secrets(persist)
    env = {
        **os.environ,
        "PORT": str(OMNI_PORT),
        "DATA_DIR": str(DATA_DIR),
        "INITIAL_PASSWORD": admin_password,
        "JWT_SECRET": sec["JWT_SECRET"],
        "API_KEY_SECRET": sec["API_KEY_SECRET"],
        "REQUIRE_API_KEY": "false",  # OmniRoute chỉ nhận kết nối nội bộ; chỉ backend được đưa ra Internet
        "NEXT_TELEMETRY_DISABLED": "1",
    }
    say("Đang khởi động OmniRoute (lần đầu có thể mất 1-2 phút)...")
    p, log = _spawn("omniroute", ["omniroute"], env, "omniroute.log")
    _wait_port(OMNI_PORT, p, "OmniRoute", log)
    say(f"OmniRoute đã chạy ở cổng {OMNI_PORT} (chỉ nội bộ, không công khai).")


def start_backend(site_password, allowed_origin):
    _kill("backend")
    if _port_open(BACK_PORT):
        raise SetupError(f"Cổng {BACK_PORT} đang bị chiếm. Chạy ô 'Tắt dịch vụ' rồi thử lại.")
    import secrets as _s

    env = {
        **os.environ,
        "HOST": "127.0.0.1",
        "PORT": str(BACK_PORT),
        "OMNIROUTE_URL": f"http://127.0.0.1:{OMNI_PORT}",
        "SITE_PASSWORD": site_password,
        "ALLOWED_ORIGIN": allowed_origin,
        "SESSION_SECRET": _s.token_hex(32),
    }
    p, log = _spawn("backend", ["node", str(BACKEND_DIR / "server.mjs")], env, "backend.log")
    _wait_port(BACK_PORT, p, "Backend", log, timeout=30)
    say(f"Backend đã chạy ở cổng {BACK_PORT} (đã bật mật khẩu và giới hạn nguồn truy cập).")


def _http_json(url, headers=None, timeout=10):
    req = urllib.request.Request(url, headers=headers or {})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode())


NGROK_HINTS = {
    "ERR_NGROK_105": "Authtoken ngrok không đúng. Lấy lại token tại dashboard.ngrok.com/get-started/your-authtoken.",
    "ERR_NGROK_107": "Authtoken ngrok không đúng hoặc đã bị thu hồi.",
    "ERR_NGROK_108": "Tài khoản ngrok đang chạy một phiên khác. Tắt phiên cũ tại dashboard.ngrok.com/agents (hoặc chạy ô 'Tắt dịch vụ' ở notebook cũ).",
    "ERR_NGROK_4018": "Cần đăng ký tài khoản ngrok và dùng authtoken.",
}


def start_tunnel(authtoken, domain="", allowed_origin=""):
    _kill("ngrok")
    env = {**os.environ, "NGROK_AUTHTOKEN": authtoken}
    cmd = ["ngrok", "http", str(BACK_PORT), "--log=stdout", "--log-format=json"]
    if domain:
        cmd += ["--url", "https://" + domain.replace("https://", "").strip("/")]
    p, log = _spawn("ngrok", cmd, env, "ngrok.log")
    url = None
    t0 = time.time()
    while time.time() - t0 < 40 and not url:
        if p.poll() is not None:
            break
        try:
            tunnels = _http_json(NGROK_API + "/api/tunnels", timeout=2).get("tunnels", [])
            https = [t["public_url"] for t in tunnels if t.get("public_url", "").startswith("https://")]
            url = (https or [t["public_url"] for t in tunnels] or [None])[0]
        except Exception:  # noqa: BLE001
            pass
        time.sleep(1)
    if not url:
        text = _tail(log, 40)
        hint = next((h for code, h in NGROK_HINTS.items() if code in text), "")
        raise SetupError("Không tạo được đường hầm ngrok." + (f"\nGợi ý: {hint}" if hint else "") + f"\nLog cuối:\n{text}")
    # Kiểm tra từ bên ngoài qua đúng địa chỉ công khai
    try:
        h = _http_json(url + "/api/health", headers={"ngrok-skip-browser-warning": "1", "Origin": allowed_origin} if allowed_origin else {"ngrok-skip-browser-warning": "1"}, timeout=15)
        ok = bool(h.get("ok"))
    except Exception:  # noqa: BLE001
        ok = False
    _state["url"] = url
    return url, ok


# ---------------------------------------------------------------- Lệnh cho notebook
def normalize_origin(text):
    text = (text or "").strip()
    if not text:
        raise SetupError("Chưa nhập ALLOWED_ORIGIN. Ví dụ: https://tenban.github.io (tên GitHub của bạn).")
    if "://" not in text:
        text = "https://" + text
    u = urllib.parse.urlparse(text)
    if u.scheme != "https" or not u.hostname or "." not in u.hostname:
        raise SetupError("ALLOWED_ORIGIN phải có dạng https://tenban.github.io")
    return f"https://{u.netloc}"


def start_all(allowed_origin, persist=True, ngrok_domain=""):
    origin = normalize_origin(allowed_origin)
    _state.update(persist=bool(persist), origin=origin)
    if not shutil.which("omniroute") or not shutil.which("ngrok") or not (BACKEND_DIR / "server.mjs").exists():
        raise SetupError("Chưa cài đặt xong. Hãy chạy ô 'Bước 1 - Cài đặt' trước.")
    if persist:
        say("Gắn Google Drive để lưu cấu hình OmniRoute qua các phiên (Google sẽ hỏi quyền một lần)...")
        mount_drive()
    say("\nNhập thông tin bí mật (không hiển thị lại). Có thể lưu sẵn trong Colab Secrets (biểu tượng chìa khóa) để khỏi nhập mỗi lần:")
    site_pw = _ask("SITE_PASSWORD", "1/3 Mật khẩu chia sẻ cho bạn bè vào website (>= 8 ký tự)", 8)
    admin_pw = _ask("OMNIROUTE_ADMIN_PASSWORD", "2/3 Mật khẩu bảng điều khiển OmniRoute (chỉ bạn dùng, >= 8 ký tự)", 8)
    ngrok_token = _ask("NGROK_AUTHTOKEN", "3/3 ngrok authtoken", 10)
    say()
    start_omniroute(admin_pw, bool(persist))
    start_backend(site_pw, origin)
    url, ok = start_tunnel(ngrok_token, ngrok_domain, origin)
    if persist:
        threading.Thread(target=_auto_backup_loop, daemon=True).start()
        try:
            (DRIVE_DIR / "last_url.txt").write_text(url)
        except OSError:
            pass
    say("\n" + "=" * 62)
    say("ĐỊA CHỈ DỊCH VỤ: " + url)
    say("=" * 62)
    if ok:
        say("Đã kiểm tra: backend phản hồi qua địa chỉ công khai.")
    else:
        say("CẢNH BÁO: chưa xác nhận được backend qua địa chỉ công khai. Chạy ô 'Kiểm tra trạng thái' sau ít giây.")
    say("\nViệc cần làm:")
    say(f"  1. Mở file docs/config.js trên GitHub, đặt BACKEND_URL: \"{url}\" (chỉ cần làm khi địa chỉ này thay đổi).")
    say("  2. Lần đầu: chạy ô 'Mở bảng điều khiển OmniRoute' để thêm Claude/các nhà cung cấp.")
    say("  3. Gửi cho bạn bè địa chỉ website GitHub Pages và mật khẩu ở bước 1/3.")


def open_dashboard():
    try:
        from google.colab import output  # type: ignore

        say("Đang mở bảng điều khiển OmniRoute (đăng nhập bằng mật khẩu 2/3 bạn đã đặt)...")
        output.serve_kernel_port_as_window(OMNI_PORT)
    except Exception as e:  # noqa: BLE001
        say(f"Không mở tự động được ({type(e).__name__}). Trong Colab, chạy:\n  from google.colab import output; output.serve_kernel_port_as_iframe({OMNI_PORT})")


def status():
    say("Tiến trình:")
    for name in ("omniroute", "backend", "ngrok"):
        p = _procs.get(name)
        alive = bool(p) and p.poll() is None
        say(f"  {name:10s}: {'đang chạy' if alive else 'KHÔNG chạy'}")
    say(f"Cổng nội bộ: OmniRoute {OMNI_PORT}={'mở' if _port_open(OMNI_PORT) else 'đóng'}, backend {BACK_PORT}={'mở' if _port_open(BACK_PORT) else 'đóng'}")
    url = _state.get("url")
    if url:
        try:
            hdr = {"ngrok-skip-browser-warning": "1"}
            if _state.get("origin"):
                hdr["Origin"] = _state["origin"]
            _http_json(url + "/api/health", headers=hdr, timeout=15)
            say(f"Địa chỉ công khai {url}: phản hồi OK")
        except Exception as e:  # noqa: BLE001
            say(f"Địa chỉ công khai {url}: KHÔNG phản hồi ({type(e).__name__})")
    else:
        say("Chưa có địa chỉ công khai (chưa chạy ô Khởi động).")
    for name in ("backend", "omniroute"):
        say(f"\n--- log {name} (10 dòng cuối) ---\n{_tail(LOG_DIR / (name + '.log'), 10)}")


def stop():
    if _state.get("persist"):
        try:
            backup_config()
        except Exception as e:  # noqa: BLE001
            say(f"Không sao lưu được cấu hình ({type(e).__name__}).")
    _state["persist"] = False
    for name in ("ngrok", "backend", "omniroute"):
        _kill(name)
    # Dọn cả tiến trình mồ côi (ví dụ sau khi khởi động lại kernel Colab làm mất danh sách tiến trình):
    # tìm theo cổng đang lắng nghe, không theo tên, để không tắt nhầm tiến trình khác.
    for port in (BACK_PORT, OMNI_PORT):
        for pid in _pids_listening(port):
            try:
                os.kill(pid, signal.SIGTERM)
            except OSError:
                pass
    subprocess.run(["pkill", "-x", "ngrok"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(1)
    say("Đã tắt toàn bộ dịch vụ.")
'''

(BASE/"backend"/"server.mjs").write_text(SERVER_MJS, encoding="utf-8")
pathlib.Path("/content/tp_omniai.py").write_text(TP_HELPER, encoding="utf-8")
sys.path.insert(0, "/content")
import importlib, tp_omniai; importlib.reload(tp_omniai)
try:
    tp_omniai.install_runtime(OMNIROUTE_VERSION)
except tp_omniai.SetupError as e:
    print("LỖI:", e)

In [ ]:
#@title Bước 2 - Khởi động TP OmniAI {display-mode: "form"}
#@markdown Địa chỉ GitHub Pages của bạn, chỉ cần phần tên miền (ví dụ `https://tenban.github.io`):
ALLOWED_ORIGIN = ""  #@param {type:"string"}
#@markdown Lưu cấu hình OmniRoute (tài khoản Claude, combo...) vào Google Drive để lần sau khỏi nhập lại:
PERSIST_TO_DRIVE = True  #@param {type:"boolean"}
#@markdown Tên miền ngrok cố định của bạn (để trống nếu dùng tên miền mặc định ngrok cấp):
NGROK_DOMAIN = ""  #@param {type:"string"}
import tp_omniai
try:
    tp_omniai.start_all(ALLOWED_ORIGIN, PERSIST_TO_DRIVE, NGROK_DOMAIN)
except tp_omniai.SetupError as e:
    print("\nLỖI:", e)

### Lần đầu: thêm Claude vào OmniRoute
Chạy ô dưới để mở bảng điều khiển OmniRoute (đăng nhập bằng *mật khẩu bảng điều khiển* bạn đã nhập ở Bước 2). Thêm nhà cung cấp Claude (và Gemini/GPT/DeepSeek nếu muốn), rồi tạo một **Combo** ưu tiên Claude, dự phòng các model còn lại. Combo/`auto` sẽ tự xuất hiện trong danh sách model của website.

In [ ]:
#@title Mở bảng điều khiển OmniRoute {display-mode: "form"}
import tp_omniai
tp_omniai.open_dashboard()

In [ ]:
#@title Kiểm tra trạng thái {display-mode: "form"}
import tp_omniai
tp_omniai.status()

In [ ]:
#@title Sao lưu cấu hình OmniRoute vào Google Drive (tự động mỗi 10 phút nếu đã bật lưu) {display-mode: "form"}
import tp_omniai
tp_omniai.backup_config()

In [ ]:
#@title Tắt dịch vụ {display-mode: "form"}
import tp_omniai
tp_omniai.stop()